In [1]:
import duckdb

con = duckdb.connect()
con.execute("CREATE VIEW inter AS SELECT * FROM '../data/processed/interactions.parquet'")

print(con.execute("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT uid) AS n_students,
           COUNT(DISTINCT qid) AS n_questions,
           ROUND(AVG(response), 3) AS overall_accuracy
    FROM inter
""").df())

    n_rows  n_students  n_questions  overall_accuracy
0  4446374       14453         7618             0.795


In [2]:
print(con.execute("""
    SELECT fold, COUNT(*) AS n_rows, ROUND(AVG(response), 3) AS accuracy
    FROM inter
    GROUP BY fold ORDER BY fold
""").df())

   fold  n_rows  accuracy
0     0  891277     0.794
1     1  887363     0.796
2     2  889980     0.791
3     3  888493     0.797
4     4  889261     0.794


In [3]:
print(con.execute("""
    SELECT uid, COUNT(DISTINCT fold) AS n_folds
    FROM inter
    GROUP BY uid
    HAVING COUNT(DISTINCT fold) > 1
    LIMIT 5
""").df())

Empty DataFrame
Columns: [uid, n_folds]
Index: []


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

VAL = 0
GLOBAL = 0.795

feat = con.execute(f"""
    WITH item AS (
        SELECT qid, AVG(response) AS item_acc, COUNT(*) AS item_n
        FROM inter WHERE fold <> {VAL}
        GROUP BY qid
    ),
    hist AS (
        SELECT *,
               AVG(response) OVER (
                   PARTITION BY seq_id ORDER BY pos
                   ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
               ) AS prev_acc
        FROM inter
    )
    SELECT h.fold, h.response, h.pos,
           COALESCE(i.item_acc, {GLOBAL}) AS item_acc,
           COALESCE(h.prev_acc, {GLOBAL}) AS prev_acc
    FROM hist h LEFT JOIN item i USING (qid)
""").df()

train = feat[feat.fold != VAL]
val = feat[feat.fold == VAL]

auc_a = roc_auc_score(val.response, val.item_acc)
print("기준선 A (문항 평균만) AUC:", round(auc_a, 4))

cols = ["item_acc", "prev_acc", "pos"]
lr = LogisticRegression(max_iter=1000).fit(train[cols], train.response)
auc_b = roc_auc_score(val.response, lr.predict_proba(val[cols])[:, 1])
print("기준선 B (로지스틱) AUC:", round(auc_b, 4))

기준선 A (문항 평균만) AUC: 0.7316
기준선 B (로지스틱) AUC: 0.7857


In [ ]:
import duckdb
con = duckdb.connect()
print(con.execute("DESCRIBE SELECT * FROM '../data/processed/interactions.parquet'").df())